# Zonostrophy comparison

Generate two trajectories with the same default parameters, seed, and resolution, varying only the energy-injection parameter. The cases are ordered and labeled by the measured Laura Cope $R_\\beta$ computed from each trajectory, rather than assuming an injection change guarantees the ordering. The code uses `scripts.generate_trajectory.main(cfg)` and saves both trajectories, metadata, and a side-by-side GIF under `outputs/zonostrophy_comparison/`.

The injection multipliers below are intentionally far apart for an exploratory comparison. Each run uses the configured `plotting.nsteps`, so running the cell may take a while.

In [1]:
import contextlib
import io
import json
import sys
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation, PillowWriter
from omegaconf import OmegaConf

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'config' / 'default.yaml').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not find config/default.yaml above the notebook directory')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from model.core.model import QGM
from model.ML.utils.coarsen import coarsen
from scripts.generate_trajectory import main as generate_trajectory_main

base_cfg = OmegaConf.load(project_root / 'config' / 'default.yaml')
output_dir = project_root / 'outputs' / 'zonostrophy_comparison'
output_dir.mkdir(parents=True, exist_ok=True)
base_epsilon = float(base_cfg.params.epsilon)
injection_factors = {
    'lower_injection': 0.01,
    'higher_injection': 100.0,
}

def run_case(name, injection_factor):
    cfg = OmegaConf.create(OmegaConf.to_container(base_cfg, resolve=True))
    cfg.params.epsilon = base_epsilon * injection_factor
    cfg.filepaths.out_dir = str(output_dir / name / 'diagnostics')
    Path(cfg.filepaths.out_dir).mkdir(parents=True, exist_ok=True)

    # main() returns the coarsened PV trajectory; its stdout includes an older,
    # differently defined R_beta, so report the requested formula below instead.
    with contextlib.redirect_stdout(io.StringIO()):
        q = np.asarray(generate_trajectory_main(cfg), dtype=np.float32)
    if q.ndim != 4 or q.shape[0] == 0:
        raise RuntimeError('Expected a nonempty (time, layer, y, x) trajectory')
    if not np.all(np.isfinite(q)):
        raise FloatingPointError('Generated trajectory contains non-finite values')

    params = dict(OmegaConf.to_container(cfg.params, resolve=True))
    params['dt'] = float(cfg.plotting.dt)
    if bool(cfg.plotting.auto_dt):
        raw_model = QGM({**params, 'nx': int(params['hr_nx'])})
        initial_state = raw_model.initialise(
            jax.random.PRNGKey(int(params.get('seed', 0))),
            n_jets=params.get('n_jets'),
            verbose=True,
        )
        dt_model = float(
            raw_model.estimate_cfl_dt(initial_state, cfl=float(cfg.plotting.cfl))
        )
        params['dt'] = dt_model * raw_model.time_scale

    hr_model = QGM({**params, 'nx': int(params['hr_nx'])})
    lr_model = coarsen(hr_model, int(params['nx']))
    sample_indices = np.linspace(0, q.shape[0] - 1, min(64, q.shape[0]), dtype=int)
    velocity_energy = []
    for frame_index in sample_indices:
        state = lr_model.set_initial(
            lr_model.real_to_spectral(q[frame_index]),
            _q_shape=(int(lr_model.ny), int(lr_model.nx)),
        )
        full_state = lr_model.get_full_state(state)
        velocity_energy.append(
            np.asarray(full_state.u) ** 2 + np.asarray(full_state.v) ** 2
        )

    u_rms = float(np.sqrt(np.mean(np.stack(velocity_energy))))
    beta = float(lr_model.beta)
    epsilon_injection = float(lr_model.epsilon)
    if u_rms <= 0 or beta <= 0 or epsilon_injection <= 0:
        raise ValueError('R_beta requires positive U_rms, beta, and energy injection')

    # Laura Cope convention supplied for this comparison; all inputs are in
    # the model's internally consistent nondimensional units.
    r_beta = (
        np.sqrt(u_rms) * beta ** 0.1
        / (np.sqrt(2.0) * epsilon_injection ** 0.2)
    )
    lr_dt_seconds = float(lr_model.dt * hr_model.time_scale)
    return {
        'name': name,
        'injection_factor': injection_factor,
        'epsilon_configured': float(cfg.params.epsilon),
        'epsilon_model': epsilon_injection,
        'u_rms': u_rms,
        'beta': beta,
        'r_beta': float(r_beta),
        'dt_seconds': lr_dt_seconds,
        'q': q,
    }

cases = [run_case(name, factor) for name, factor in injection_factors.items()]
cases.sort(key=lambda case: case['r_beta'], reverse=True)
strong, weak = cases

for label, case in (('Higher R_beta (stronger)', strong), ('Lower R_beta (weaker)', weak)):
    print(
        '{}: epsilon_config={:.4g}, epsilon_model={:.4g}, '
        'U_rms={:.5g}, beta={:.5g}, R_beta={:.5g}, shape={}'.format(
            label,
            case['epsilon_configured'],
            case['epsilon_model'],
            case['u_rms'],
            case['beta'],
            case['r_beta'],
            case['q'].shape,
        )
    )

np.savez_compressed(
    output_dir / 'trajectory_pair.npz',
    q_higher_r_beta=strong['q'],
    q_lower_r_beta=weak['q'],
)
metadata = {
    'definition': 'R_beta = sqrt(U_rms) * beta**0.1 / (sqrt(2) * epsilon_injection**0.2)',
    'units': 'model nondimensional units',
    'higher_r_beta': {key: value for key, value in strong.items() if key != 'q'},
    'lower_r_beta': {key: value for key, value in weak.items() if key != 'q'},
}
with (output_dir / 'metadata.json').open('w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2)

nframes = min(strong['q'].shape[0], weak['q'].shape[0])
stride = max(1, int(base_cfg.plotting.cadence))
frame_indices = np.arange(0, nframes, stride, dtype=int)
if frame_indices[-1] != nframes - 1:
    frame_indices = np.append(frame_indices, nframes - 1)
pv_values = np.concatenate(
    (strong['q'][frame_indices, 0].ravel(), weak['q'][frame_indices, 0].ravel())
)
vmax = float(np.nanpercentile(np.abs(pv_values), 99.5))
if vmax == 0:
    vmax = 1.0

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
images = [
    axes[0].imshow(strong['q'][0, 0], origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax),
    axes[1].imshow(weak['q'][0, 0], origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax),
]
axes[0].set_title('Higher R_beta (stronger)')
axes[1].set_title('Lower R_beta (weaker)')
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(images[0], ax=axes, label='Potential vorticity')
fig.suptitle('R_beta={:.3g} vs {:.3g}'.format(strong['r_beta'], weak['r_beta']))

def update(animation_index):
    frame_index = int(frame_indices[animation_index])
    images[0].set_data(strong['q'][frame_index, 0])
    images[1].set_data(weak['q'][frame_index, 0])
    days_strong = frame_index * strong['dt_seconds'] / 86400.0
    days_weak = frame_index * weak['dt_seconds'] / 86400.0
    axes[0].set_xlabel('{:.1f} days'.format(days_strong))
    axes[1].set_xlabel('{:.1f} days'.format(days_weak))
    return images

animation = FuncAnimation(fig, update, frames=len(frame_indices), interval=100, blit=False)
gif_path = output_dir / 'zonostrophy_comparison.gif'
animation.save(gif_path, writer=PillowWriter(fps=8))
plt.close(fig)
print('Saved GIF:', gif_path)
print('Saved trajectories:', output_dir / 'trajectory_pair.npz')

2026-11-23 12:53:25,901 INFO scripts.generate_trajectory: Requested device: cpu, using device: CPU
2026-11-23 12:57:43,894 INFO scripts.generate_trajectory: Fine timestep is 2e+03s and coarsened timestep is 3.2e+04s. Model spinup for 0.00 eddy turnover times (~0 high-res steps). Training horizon is 2000 low-res steps (~740.74 days). Validation plotting window is 2000 steps (~740.74 days).


KeyboardInterrupt: 